# DeepFashion-MultiModal: captions and tags

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/deepfashion_captions.ipynb)

Every photo of DeepFashion-MultiModal (44,096) gets a caption from the stock Qwen3.5-4B, asked with `captions.FASHION_ASK` and the photo's own textual description as the helper line, and booru tags from the animetimm EVA-giant tagger (`captions.run`). The rows ship to `extractions/<stamp>/` of the dataset `AbstractPhil/qwen-thinking-images` while the run goes; after a lost runtime, set `RESUME` in C3 to the shipped folder and run C1-C3 again.

The photos and the descriptions come from the dataset's official links ([github.com/yumingj/DeepFashion-MultiModal](https://github.com/yumingj/DeepFashion-MultiModal)), for non-commercial research use under its agreement. A row holds the file name, our caption, our tags and how much of the caption repeats the description word for word; not the dataset's images or descriptions.

Runtime: a GPU (the G4, RTX PRO 6000) and the `HF_TOKEN` secret with write access to the dataset repo and access to the gated animetimm tagger. Run C1, restart the session (Runtime > Restart session), then run C2 and C3.

In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.41", "transformers>=5.0", "accelerate>=1.0", "timm>=1.0")

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (the run still works, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

print("installed; now Runtime > Restart session, then run C2 and C3")

In [ ]:
# C2: the photos and the descriptions, from the dataset's official links (non-commercial research use)
import os, re, subprocess, sys, time, zipfile
import requests

DATA = "/content/deepfashion"
LINKS = {"images.zip": "1U2PljA7NE57jcSSzPs21ZurdIPXdYZtN", "captions.json": "1d1TRm8UMcQhZCb6HpPo8l3OPEin4Ztk2"}   # Google Drive file ids
URL = "https://drive.usercontent.google.com/download"
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}
os.makedirs(DATA, exist_ok=True)

def good(path, name):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return False
    if name.endswith(".zip"):
        return zipfile.is_zipfile(path)
    with open(path, "rb") as fh:
        return fh.read(1) in (b"{", b"[")

def stream(session, params, path, name):
    # the file into path; or, when Drive answers with a page instead, that page's text
    t0, got, step = time.time(), 0, 512 << 20
    with session.get(URL, params=params, stream=True, timeout=60, headers=UA) as r:
        r.raise_for_status()
        if "text/html" in r.headers.get("content-type", ""):
            return r.text
        total = int(r.headers.get("content-length") or 0)
        with open(path + ".part", "wb") as fh:
            for block in r.iter_content(8 << 20):
                fh.write(block)
                if (got + len(block)) // step > got // step:
                    rate = (got + len(block)) / max(time.time() - t0, 1e-9)
                    left = f", {(total - got) / rate:.0f} s left" if total else ""
                    print(f"  {name}: {(got + len(block)) / 1e9:.2f} of {total / 1e9:.2f} GB, {rate / 1e6:.0f} MB/s{left}", flush=True)
                got += len(block)
    os.replace(path + ".part", path)
    print(f"{name}: {got / 1e9:.2f} GB in {time.time() - t0:.0f} s")
    return None

def fetch(name, file_id):
    path = os.path.join(DATA, name)
    if good(path, name):
        print(f"{name}: already here, {os.path.getsize(path) / 1e9:.2f} GB")
        return path
    if os.path.exists(path):
        os.remove(path)
    s = requests.Session()
    page = stream(s, {"id": file_id, "export": "download", "confirm": "t"}, path, name)
    if page:            # Drive's warning page carries a form with the confirm token (and a uuid) for the download itself
        form = dict(re.findall(r'name="([^"]+)" value="([^"]*)"', page))
        if form.get("id"):
            page = stream(s, form, path, name) or None
    if page or not good(path, name):            # last: gdown, which knows Drive's other pages
        subprocess.call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown
        gdown.download(id=file_id, output=path, quiet=False)
    if not good(path, name):
        text = " ".join(re.sub(r"<[^>]+>", " ", page or "").split())
        raise RuntimeError(f"{name}: Drive did not give the file; its page says: {text[:500]}")
    return path

IMAGES, DESCRIPTIONS = fetch("images.zip", LINKS["images.zip"]), fetch("captions.json", LINKS["captions.json"])
print(sum(1 for m in zipfile.ZipFile(IMAGES).namelist() if m.lower().endswith(".jpg")), "photos")

In [ ]:
# C3: caption and tag every photo; the rows ship to the dataset repo while the run goes
from qwen_symbolic_thinking.captions import FASHION_ASK, run

RESUME = None   # after a lost runtime: the shipped folder, e.g. "extractions/20261104_230000_deepfashion_mm_captions"
TAG_OMIT = ("short hair", "see-through", "see-through sleeves", "see-through shirt", "see-through clothes", "hair bun",
            "single hair bun", "double bun", "parted lips", "lips", "realistic", "photorealistic", "asian")   # the reviewer's leave-out list

summary = run(IMAGES, DESCRIPTIONS, model_id="Qwen/Qwen3.5-4B", ask=FASHION_ASK, batch=64, chunk=512, max_new_tokens=160,
              tagger="animetimm/eva_giant_patch14_560.dbv4-full", tag_omit=TAG_OMIT, tag_min_precision=0.4,
              tag="deepfashion_mm_captions", resume=RESUME)
print(summary["path_in_repo"])